In [121]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import random
%matplotlib inline
%matplotlib ipympl
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [122]:
data = pd.read_csv("data.csv")
events = pd.read_csv("events.csv")

print("DATA columns:")
print(data.columns.tolist())

print("\nEVENTS columns:")
print(events.columns.tolist())

DATA columns:
['order_id', 'user_id', 'status', 'gender', 'created_at', 'returned_at', 'shipped_at', 'delivered_at', 'num_of_item', 'product_id', 'inventory_item_id', 'sale_price', 'id', 'first_name', 'last_name', 'email', 'age', 'state', 'street_address', 'postal_code', 'city', 'country', 'traffic_source', 'user_geom', 'cost', 'category', 'brand', 'retail_price', 'department', 'sku', 'distribution_center_id', 'sold_at', 'product_category', 'product_name', 'product_brand', 'product_retail_price', 'product_department', 'product_sku', 'product_distribution_center_id', 'distribution_center_geom', 'order_item_id', 'delivery_longitude', 'delivery_latitude', 'warehouse_name', 'warehouse_longitude', 'warehouse_latitude', 'is_loyal', 'product_name_clean', 'customer_review']

EVENTS columns:
['id', 'user_id', 'sequence_number', 'session_id', 'created_at', 'ip_address', 'city', 'state', 'postal_code', 'browser', 'traffic_source', 'uri', 'event_type']


In [123]:
data['status'].unique()

array(['Shipped', 'Complete', 'Cancelled', 'Returned', 'Processing'],
      dtype=object)

In [124]:
import pandas as pd


class SimplePreprocessor:
    def __init__(self):
        self.columns = [
            'user_id',
            'order_id',
            'status',
            'category',
            'brand',
            'age',
            'is_loyal',
            'gender',
            'country',
            'cost',
            'sale_price',
            'created_at',
            'product_id'
        ]

        self.valid_status = ['Shipped', 'Complete', 'Cancelled', 'Returned', 'Processing']

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        print(f"Initial rows: {len(df)}")

        # 1. оставить только нужные колонки
        df = df[self.columns]
        print(f"After column selection: {len(df)}")

        # 2. удалить дубликаты
        df = df.drop_duplicates()
        print(f"After drop_duplicates: {len(df)}")

        # 3. удалить строки с NaN
        df = df.dropna()
        print(f"After dropna: {len(df)}")

        # 4. фильтр статусов
        df = df[df['status'].isin(self.valid_status)]
        print(f"After status filter: {len(df)}")

        return df.reset_index(drop=True)

In [125]:
prep = SimplePreprocessor()

df_clean = prep.transform(data)

Initial rows: 545778
After column selection: 545778
After drop_duplicates: 181922
After dropna: 181738
After status filter: 181738


In [126]:
data[['cost', 'sale_price']]

,cost,sale_price
0,16.330051,32.990002
1,19.824570,39.570000
2,96.117338,221.979996
3,13.750000,25.000000
4,11.400000,25.000000
...,...,...
545773,15.100000,25.000000
545774,14.297000,29.000000
545775,14.545150,29.990000
545776,22.551389,59.189999


In [127]:
import pandas as pd
import numpy as np

df = df_clean.copy()

# =========================
# 1. ФИЛЬТР
# =========================
df = df[df['status'].isin(['Complete', 'Returned'])].copy()

# =========================
# 2. МАРЖА
# =========================
df['margin'] = df['sale_price'] - df['cost']

# =========================
# 3. АГРЕГАЦИЯ
# =========================
agg = df.groupby(['category', 'brand']).agg(
    complete_cnt=('status', lambda x: (x == 'Complete').sum()),
    returned_cnt=('status', lambda x: (x == 'Returned').sum()),
    avg_margin=('margin', 'mean'),
    avg_price=('sale_price', 'mean')
).reset_index()

# =========================
# 4. ФИЧИ
# =========================
agg['purchases'] = agg['complete_cnt']

agg['buyout_rate'] = agg['complete_cnt'] / (
    agg['complete_cnt'] + agg['returned_cnt'] + 1e-9
)

# Байесовское сглаживание
global_rate = df[df['status'] == 'Complete'].shape[0] / len(df)
agg['buyout_smooth'] = (
    agg['complete_cnt'] + 10 * global_rate
) / (agg['complete_cnt'] + agg['returned_cnt'] + 10)

agg['expected_margin'] = agg['avg_margin'] * agg['buyout_smooth']

# =========================
# 5. ЛОГ ПОПУЛЯРНОСТЬ
# =========================
agg['log_purchases'] = np.log1p(agg['purchases'])

# =========================
# 6. НОРМАЛИЗАЦИЯ
# =========================
def normalize(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

agg['margin_norm'] = normalize(agg['avg_margin'])
agg['buyout_norm'] = normalize(agg['buyout_smooth'])
agg['popularity_norm'] = normalize(agg['log_purchases'])

# =========================
# 7. НАДЕЖНОСТЬ
# =========================
agg['is_reliable'] = agg['purchases'] > 20

# =========================
# 8. ACTUAL SCORE
# =========================
agg['actual_score'] = (
    0.5 * agg['margin_norm'] +
    0.3 * agg['buyout_norm'] +
    0.2 * agg['popularity_norm']
)

agg.loc[agg['is_reliable'], 'actual_score'] *= 1.2

# =========================
# 9. РЕЙТИНГ (для UI)
# =========================
agg['rating'] = (agg['buyout_smooth'] * 5).round(2)

# =========================
# 10. СОРТИРОВКИ
# =========================
by_price_des = agg.sort_values(['category', 'avg_price'], ascending=[True, False])
by_price_asc = agg.sort_values(['category', 'avg_price'], ascending=[True, True])
by_popularity = agg.sort_values(['category', 'purchases'], ascending=[True, False])
by_rating = agg.sort_values(['category', 'rating'], ascending=[True, False])
by_actual = agg.sort_values(['category', 'actual_score'], ascending=[True, False])

# =========================
# RESULT
# =========================
by_price_desc.head(20)

,category,brand,complete_cnt,returned_cnt,avg_margin,avg_price,purchases,buyout_rate,buyout_smooth,expected_margin,log_purchases,margin_norm,buyout_norm,popularity_norm,boost,actual_score,rating,rating_boosted,top_products
335,Accessories,Tiffany,3,1,163.442500,267.500000,3,0.750000,0.725028,118.500404,1.386294,0.306764,0.642152,0.244951,1.00,0.395018,3.63,3.6300,[14200]
340,Accessories,Tom Ford,14,3,160.481206,256.369417,14,0.823529,0.783348,125.712616,2.708050,0.301205,0.768648,0.478498,1.15,0.548431,3.92,4.5080,"[28550, 28833]"
93,Accessories,Dolce & Gabbana,5,1,147.015001,243.000000,5,0.833333,0.759400,111.643130,1.791759,0.275929,0.716704,0.316594,1.00,0.416294,3.80,3.8000,"[13751, 28540]"
261,Accessories,Prada,1,0,123.519000,209.000000,1,1.000000,0.740945,91.520766,0.693147,0.231826,0.676676,0.122475,1.00,0.343411,3.70,3.7000,[13653]
213,Accessories,Maui Jim,8,4,122.446890,204.470834,8,0.666667,0.688654,84.323570,2.197225,0.229814,0.563258,0.388238,1.00,0.361532,3.44,3.4400,"[14024, 14281]"
67,Accessories,Costa Del Mar,9,2,115.473276,200.772724,9,0.818182,0.769066,88.806610,2.302585,0.216724,0.737671,0.406854,1.00,0.411034,3.85,3.8500,"[13932, 14182]"
109,Accessories,FRYE,3,0,113.652000,198.000000,3,1.000000,0.780799,88.739423,1.386294,0.213306,0.763120,0.244951,1.00,0.384579,3.90,3.9000,[13924]
138,Accessories,Gucci,9,1,113.304389,187.357997,9,0.900000,0.807520,91.495522,2.302585,0.212653,0.821076,0.406854,1.00,0.434020,4.04,4.0400,"[28665, 14162]"
252,Accessories,Persol,4,1,103.197221,168.638000,4,0.800000,0.743360,76.712640,1.609438,0.193682,0.681913,0.284379,1.00,0.358291,3.72,3.7200,"[28403, 28823]"
49,Accessories,Canada Goose,4,4,95.402138,162.456250,4,0.500000,0.619466,59.098409,1.609438,0.179050,0.413190,0.284379,1.00,0.270358,3.10,3.1000,"[14243, 29027]"


In [128]:
agg.to_csv('agg.csv')

In [129]:
import pandas as pd
import numpy as np

df = df_clean.copy()

# =========================
# 1. ФИЛЬТР
# =========================
df = df[df['status'].isin(['Complete', 'Returned'])].copy()
df = df.dropna(subset=['gender'])

# =========================
# ФУНКЦИЯ ДЛЯ ОДНОГО ПОЛА
# =========================
def build_tables(df_gender):

    df_gender = df_gender.copy()

    # -------------------------
    # МАРЖА
    # -------------------------
    df_gender['margin'] = df_gender['sale_price'] - df_gender['cost']

    # -------------------------
    # PRODUCT LEVEL
    # -------------------------
    prod = df_gender.groupby(['category', 'brand', 'product_id']).agg(
        purchases=('status', lambda x: (x == 'Complete').sum()),
        avg_price=('sale_price', 'mean')
    ).reset_index()

    prod = prod.sort_values(
        ['category', 'brand', 'purchases'],
        ascending=[True, True, False]
    )

    prod['rank'] = prod.groupby(['category', 'brand']).cumcount() + 1

    # только бренды с >= 2 товарами
    brand_counts = prod.groupby(['category', 'brand'])['product_id'].count().reset_index()
    valid_brands = brand_counts[brand_counts['product_id'] >= 2][['category', 'brand']]

    prod = prod.merge(valid_brands, on=['category', 'brand'])

    # топ-2 товара
    top_prod = prod[prod['rank'] <= 2]

    top_prod = top_prod.pivot_table(
        index=['category', 'brand'],
        columns='rank',
        values=['product_id', 'avg_price'],
        aggfunc='first'
    )

    top_prod.columns = [
        f"{col[0].replace('product_id','product').replace('avg_price','price')}_{col[1]}"
        for col in top_prod.columns
    ]

    top_prod = top_prod.reset_index()

    # -------------------------
    # BRAND LEVEL
    # -------------------------
    df_gender = df_gender.merge(valid_brands, on=['category', 'brand'])

    agg = df_gender.groupby(['category', 'brand']).agg(
        complete_cnt=('status', lambda x: (x == 'Complete').sum()),
        returned_cnt=('status', lambda x: (x == 'Returned').sum()),
        avg_margin=('margin', 'mean'),
        avg_price=('sale_price', 'mean')
    ).reset_index()

    agg['purchases'] = agg['complete_cnt']

    # -------------------------
    # BUYOUT
    # -------------------------
    agg['buyout_rate'] = agg['complete_cnt'] / (
        agg['complete_cnt'] + agg['returned_cnt'] + 1e-9
    )

    global_rate = df_gender[df_gender['status'] == 'Complete'].shape[0] / len(df_gender)

    agg['buyout_smooth'] = (
        agg['complete_cnt'] + 10 * global_rate
    ) / (agg['complete_cnt'] + agg['returned_cnt'] + 10)

    # -------------------------
    # МЕТРИКИ
    # -------------------------
    agg['expected_margin'] = agg['avg_margin'] * agg['buyout_smooth']
    agg['log_purchases'] = np.log1p(agg['purchases'])

    # -------------------------
    # НОРМАЛИЗАЦИЯ
    # -------------------------
    def normalize(s):
        return (s - s.min()) / (s.max() - s.min() + 1e-9)

    agg['margin_norm'] = normalize(agg['avg_margin'])
    agg['buyout_norm'] = normalize(agg['buyout_smooth'])
    agg['popularity_norm'] = normalize(agg['log_purchases'])

    # -------------------------
    # БУСТ
    # -------------------------
    agg['boost'] = np.where(agg['purchases'] > 10, 1.1, 1.0)

    # -------------------------
    # ACTUAL SCORE
    # -------------------------
    agg['actual_score'] = (
        0.5 * agg['margin_norm'] +
        0.5 * agg['buyout_norm']
    ) * agg['boost']

    # -------------------------
    # РЕЙТИНГ (2–5)
    # -------------------------
    base = agg['buyout_smooth']

    min_b = base.min()
    max_b = base.max()

    agg['rating'] = 2 + 3 * (base - min_b) / (max_b - min_b + 1e-9)
    agg['rating'] = agg['rating'].round(2)

    agg['rating_boosted'] = agg['rating'] * agg['boost']

    # -------------------------
    # MERGE ТОВАРОВ
    # -------------------------
    agg = agg.merge(top_prod, on=['category', 'brand'], how='left')

    # -------------------------
    # СОРТИРОВКИ
    # -------------------------
    by_price_desc = agg.sort_values(['category', 'avg_price'], ascending=[True, False])
    by_price_asc = agg.sort_values(['category', 'avg_price'], ascending=[True, True])

    by_popularity = agg.sort_values(
        ['category', 'purchases', 'boost'], ascending=[True, False, False]
    )

    by_rating = agg.sort_values(
        ['category', 'rating_boosted'], ascending=[True, False]
    )

    by_actual = agg.sort_values(
        ['category', 'actual_score'], ascending=[True, False]
    )

    return {
        'price_desc': by_price_desc,
        'price_asc': by_price_asc,
        'popularity': by_popularity,
        'rating': by_rating,
        'actual': by_actual
    }

# =========================
# 2. РАЗДЕЛЕНИЕ ПО ПОЛУ
# =========================
df_male = df[df['gender'] == 'M']
df_female = df[df['gender'] == 'F']

male_tables = build_tables(df_male)
female_tables = build_tables(df_female)

# =========================
# RESULT
# =========================
male_tables['actual'].head(20)
female_tables['actual'].head(20)

,category,brand,complete_cnt,returned_cnt,avg_margin,avg_price,purchases,buyout_rate,buyout_smooth,expected_margin,...,buyout_norm,popularity_norm,boost,actual_score,rating,rating_boosted,price_1,price_2,product_1,product_2
64,Accessories,Oakley,26,6,77.187563,126.895625,26,0.812500,0.789283,60.922860,...,0.796680,0.597784,1.1,0.548936,4.39,4.829,83.820000,109.000000,14173,14121
12,Accessories,Carrera,14,2,49.649639,84.724373,14,0.875000,0.813458,40.387883,...,0.850127,0.491174,1.1,0.538068,4.55,5.005,85.419998,80.949997,13885,14256
15,Accessories,Coach,14,3,73.943404,123.362942,14,0.823529,0.783330,57.922063,...,0.783517,0.491174,1.1,0.536953,4.35,4.785,130.000000,119.050003,14069,14238
88,Accessories,Tom Ford,6,2,148.216589,243.003754,6,0.750000,0.730550,108.279640,...,0.666826,0.352940,1.0,0.528519,4.00,4.000,297.950012,203.949997,13758,14105
19,Accessories,Costa Del Mar,6,1,109.950484,197.521426,6,0.857143,0.773524,85.049296,...,0.761836,0.352940,1.0,0.525160,4.29,4.290,188.949997,218.949997,13932,14182
32,Accessories,Gucci,5,1,116.975849,197.463331,5,0.833333,0.759369,88.827814,...,0.730542,0.324981,1.0,0.518851,4.19,4.190,176.979996,173.940002,14162,13838
35,Accessories,HOBO,27,6,48.215611,80.202727,27,0.818182,0.794184,38.292055,...,0.807514,0.604380,1.1,0.512534,4.42,4.862,55.619999,87.940002,14250,14026
1,Accessories,180s,12,1,14.753163,24.950000,12,0.923077,0.832604,12.283549,...,0.892458,0.465219,1.1,0.510327,4.68,5.148,30.000000,24.990000,13615,13756
96,Accessories,boxed-gifts,22,3,7.508582,12.270800,22,0.880000,0.832854,6.253555,...,0.893011,0.568701,1.1,0.500038,4.68,5.148,6.990000,14.990000,13789,14245
66,Accessories,Pashmina,8,0,21.557313,36.365001,8,1.000000,0.841661,18.143954,...,0.912482,0.398522,1.0,0.482990,4.74,4.740,29.990000,45.990002,14286,13915


In [130]:
import os

os.makedirs("results", exist_ok=True)

# =========================
# MALE
# =========================
for name, table in male_tables.items():
    table.to_csv(f"results/male_{name}.csv", index=False)

# =========================
# FEMALE
# =========================
for name, table in female_tables.items():
    table.to_csv(f"results/female_{name}.csv", index=False)

In [131]:
import pandas as pd
import os

def recommend_brands(
    gender: str,
    category: str,
    n: int = 5,
    sort_type: str = "actual",
    path: str = "results",
    full: bool = False
):
    """
    Рекомендация брендов по категории и полу.

    Параметры:
    ----------
    gender : str
        'M' — мужчины
        'F' — женщины

    category : str
        Категория товара (например: 'Jeans', 'Dresses')

    n : int
        Количество брендов

    sort_type : str
        Тип сортировки:
        - 'actual'      — лучший баланс (маржа + выкуп + популярность)
        - 'price_asc'   — дешевые
        - 'price_desc'  — дорогие
        - 'popularity'  — самые популярные
        - 'rating'      — лучший выкуп

    path : str
        Путь до сохранённых таблиц

    full : bool
        True  — вернуть все колонки (для анализа)
        False — только пользовательские

    Возвращает:
    ----------
    DataFrame
    """

    gender_map = {'M': 'male', 'F': 'female'}

    if gender not in gender_map:
        raise ValueError("gender должен быть 'M' или 'F'")

    valid_sorts = ['actual', 'price_desc', 'price_asc', 'popularity', 'rating']
    if sort_type not in valid_sorts:
        raise ValueError(f"sort_type должен быть из {valid_sorts}")

    filepath = os.path.join(path, f"{gender_map[gender]}_{sort_type}.csv")

    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Файл не найден: {filepath}")

    df = pd.read_csv(filepath)

    # =========================
    # ФИЛЬТР ПО КАТЕГОРИИ
    # =========================
    df = df[df['category'] == category]

    if df.empty:
        return pd.DataFrame()

    # =========================
    # ТОП-N
    # =========================
    df = df.head(n)

    # =========================
    # FULL / SHORT MODE
    # =========================
    if not full:
        visible_cols = [
            'brand',
            'rating',
            'product_1',
            'price_1',
            'product_2',
            'price_2'
        ]
        visible_cols = [c for c in visible_cols if c in df.columns]
        df = df[visible_cols]

    # =========================
    # ЧИСТКА NaN (для UI)
    # =========================
    for col in ['product_1', 'product_2']:
        if col in df.columns:
            df[col] = df[col].fillna('')

    for col in ['price_1', 'price_2']:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    return df.reset_index(drop=True)

In [132]:
recommend_brands(
    gender='F',
    category='Accessories',
    n=10,
    sort_type='actual',
    full=False
)

,brand,rating,product_1,price_1,product_2,price_2
0,Oakley,4.39,14173,83.820000,14121,109.000000
1,Carrera,4.55,13885,85.419998,14256,80.949997
2,Coach,4.35,14069,130.000000,14238,119.050003
3,Tom Ford,4.00,13758,297.950012,14105,203.949997
4,Costa Del Mar,4.29,13932,188.949997,14182,218.949997
5,Gucci,4.19,14162,176.979996,13838,173.940002
6,HOBO,4.42,14250,55.619999,14026,87.940002
7,180s,4.68,13615,30.000000,13756,24.990000
8,boxed-gifts,4.68,13789,6.990000,14245,14.990000
9,Pashmina,4.74,14286,29.990000,13915,45.990002
